# 02 — Interactive Metrics with Plotly

Real HF data (500 jobs, 274/500 relevant for Ahmed, 90 for HR). Compare **BM25 vs Embedding (TF-IDF) vs Hybrid vs Hybrid+CE**.
Interactive: hover for P@K/R@K/MRR/nDCG + latency. Export to `docs/images` still works, but Plotly gives zoom/pan for LinkedIn carousel.

Requires `pip install plotly pandas` — already in `backend/requirements.txt` light + `requirements-full.txt`

In [ ]:
import json, pathlib
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

m = json.loads(pathlib.Path('../artifacts/metrics.json').read_text() if pathlib.Path('../artifacts/metrics.json').exists() else pathlib.Path('artifacts/metrics.json').read_text())
# fallback: try both paths
try:
    p = pathlib.Path('artifacts/metrics.json')
    if not p.exists(): p = pathlib.Path('../artifacts/metrics.json')
    m = json.loads(p.read_text())
except: m = json.loads(open('artifacts/metrics.json').read())
df = pd.DataFrame(m['methods']).T.reset_index().rename(columns={'index':'method'})
df

In [ ]:
# nDCG@10 bar — interactive
fig = px.bar(df, x='method', y='ndcg@10', color='method',
             color_discrete_map={'bm25':'#94a3b8','embedding':'#a78bfa','hybrid':'#e0f11f','hybrid+ce':'#7c3aed'},
             text='ndcg@10', title='nDCG@10 by Method (hover for details)')
fig.update_traces(texttemplate='%{text:.3f}', textposition='outside', hovertemplate='<b>%{x}</b><br>nDCG@10=%{y:.3f}<br>P@10=%{customdata[0]:.2f}<br>MRR=%{customdata[1]:.2f}')
fig.update_traces(customdata=df[['precision@10','mrr']])
fig.update_layout(showlegend=False, yaxis_range=[0,1], height=400)
fig.show()
# also save static for README
fig.write_image('../docs/images/ndcg_plotly.png')
fig.write_html('../docs/images/ndcg_plotly.html')

In [ ]:
# Quality vs Cost scatter — interactive
fig = px.scatter(df, x='latency_p50_ms', y='ndcg@10', color='method', size='ndcg@10',
                 hover_data=['precision@10','mrr'], size_max=20,
                 color_discrete_map={'bm25':'#94a3b8','embedding':'#a78bfa','hybrid':'#e0f11f','hybrid+ce':'#7c3aed'},
                 title='Quality vs Cost (p50 latency vs nDCG@10)')
for i, row in df.iterrows():
    fig.add_annotation(x=row['latency_p50_ms'], y=row['ndcg@10'], text=row['method'], showarrow=False, yshift=15)
fig.update_layout(height=400)
fig.show()
fig.write_html('../docs/images/latency_plotly.html')

In [ ]:
# P@10 / R@10 / MRR grouped — interactive
long = df.melt(id_vars='method', value_vars=['precision@10','recall@10','mrr','ndcg@10'], var_name='metric', value_name='score')
fig = px.bar(long, x='method', y='score', color='metric', barmode='group', title='P@10 / R@10 / MRR / nDCG@10 — grouped')
fig.update_layout(height=400, yaxis_range=[0,1.1])
fig.show()
fig.write_html('../docs/images/grouped_plotly.html')